# Exploratory Data Analysis

This notebook looks at the real UrbanScape SEO dataset and focuses on the questions the prompt asks: how traffic varies over time and by segment, how click and impression metrics relate, how ranking relates to traffic, and which variables are strongest in the current data.

This is intentionally exploratory: the goal is to understand the real data before formal modeling.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

path = Path('..') / 'data' / 'raw' / 'UrbanScape_Apparel_SEO_Performance_Final_Dataset.xlsx'
df = pd.read_excel(path)
df.head()

## Question 1: How does organic traffic vary over time?

We aggregate by month and inspect whether traffic shows obvious time trends.

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
monthly = df.groupby(df['Date'].dt.to_period('M'))['Organic_Traffic'].mean().to_timestamp()
monthly.plot(kind='line', figsize=(10,4))
plt.title('Organic traffic by month')
plt.ylabel('Mean organic traffic')
plt.tight_layout()

Interpretation: the line chart is used to understand whether traffic has a visible temporal pattern or variation. This helps decide whether the data supports temporal modeling or is better treated as a cross-sectional SEO dataset.

## Question 2: How do clicks and impressions relate?

In [ ]:
sns.scatterplot(data=df, x='Impressions', y='Clicks', alpha=0.5, s=10)
plt.title('Clicks vs impressions')
plt.tight_layout()

Interpretation: this compares click volume to impression volume to understand whether the dataset is dominated by traffic scale or by conversion efficiency.

## Question 3: How does CTR relate to clicks and impressions?

In [ ]:
sns.scatterplot(data=df, x='CTR (%)', y='Clicks', alpha=0.5, s=10)
plt.title('CTR vs clicks')
plt.tight_layout()

Interpretation: CTR is a derived efficiency metric and should be treated carefully because it may be mathematically dependent on click and impression counts.

## Question 4: How does average position relate to clicks and traffic?

In [ ]:
sns.scatterplot(data=df, x='Average_Position', y='Organic_Traffic', alpha=0.5, s=10)
plt.title('Average position vs organic traffic')
plt.tight_layout()

Interpretation: lower average position generally signals stronger ranking performance, but this needs to be interpreted alongside other metrics like clicks, impressions, and conversion rate.

## Question 5: Does domain authority relate to traffic?

In [ ]:
sns.scatterplot(data=df, x='Domain_Authority', y='Organic_Traffic', alpha=0.5, s=10)
plt.title('Domain authority vs organic traffic')
plt.tight_layout()

Interpretation: this comparison helps determine whether authority-style SEO metrics are predictive of organic traffic in the current dataset.

## Question 6: How do backlinks relate to ranking?

In [ ]:
sns.scatterplot(data=df, x='Backlinks', y='Average_Position', alpha=0.5, s=10)
plt.title('Backlinks vs average position')
plt.tight_layout()

Interpretation: a looser relationship may indicate that backlinks contribute to rankings but are not sufficient on their own.

## Question 7: How do technical SEO metrics relate to organic traffic?

In [ ]:
sns.scatterplot(data=df, x='Page_Load_Time (sec)', y='Organic_Traffic', alpha=0.5, s=10)
plt.title('Page load time vs organic traffic')
plt.tight_layout()

Interpretation: technical performance variables can be useful predictors but must be checked for noise, collinearity, and causality assumptions.

## Question 8: How do Core Web Vitals vary across records?

In [ ]:
for metric in ['Core_Web_Vitals_LCP (sec)', 'Core_Web_Vitals_FID (ms)', 'Core_Web_Vitals_CLS']:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[metric], bins=20, kde=True)
    plt.title(f'{metric} distribution')
    plt.tight_layout()

Interpretation: distributional analysis helps determine whether website performance metrics are within a realistic range, whether there are outliers, and what kinds of optimization tasks may be grounded in the dataset.

## Question 9: How does performance differ across device types?

In [ ]:
device_summary = df.groupby('Device Type')['Organic_Traffic'].mean().sort_values(ascending=False)
device_summary.plot(kind='bar', figsize=(8,4))
plt.title('Mean organic traffic by device type')
plt.ylabel('Organic traffic')
plt.tight_layout()

## Question 10: How does performance vary by location, media, source, and landing page?

In [ ]:
for col in ['Location', 'Media Type', 'Social Media Source', 'Top_Landing_Pages']:
    plt.figure(figsize=(8,4))
    df.groupby(col)['Organic_Traffic'].mean().sort_values(ascending=False).plot(kind='bar')
    plt.title(f'Mean organic traffic by {col}')
    plt.ylabel('Organic traffic')
    plt.tight_layout()

Interpretation: segmentation helps identify which marketing contexts or landing page patterns may be strongest for SEO performance.

## Question 11: Which features have the strongest correlations with key outcomes?

In [ ]:
numeric = df.select_dtypes(include='number')
corr = numeric.corr(numeric_only=True)
for target in ['Organic_Traffic', 'Clicks', 'Conversion_Rate (%)', 'Organic_Revenue ($)', 'Average_Position']:
    if target in corr.columns:
        print(f'\nTop relationships for {target}:')
        print(corr[target].drop(target).sort_values(ascending=False).head(10).to_string())

## Final interpretation

This dataset appears to be cross-sectional and rich enough for segmentation and predictive analysis, but it does not show a strong repeated-entity time series. Therefore, strong time-based decline detection should be treated cautiously and only after explicit entity-level validation.

The next step is to move from exploration into target definition, feature construction, and first baseline models for valid tasks such as traffic regression and high-performance classification.